In [ ]:
import os
# os.chdir("..")  # change to project root if necessary

import numpy as np
import torch
import random
import seaborn as sns
import matplotlib.pyplot as plt

attribution_map = torch.from_numpy(np.load('temp/debias/bert-base-uncased/attribution_map.npy'))

In [ ]:
index1 = random.randint(0, len(attribution_map))
index2 = random.randint(0, len(attribution_map))

index1, index2 = 540, 1269

maps = [attribution_map[index1], attribution_map[index2]]
titles = [f"Example {index1}", f"Example {index2}"]

# 对两个矩阵做池化
pooled_maps = []
for random_map in maps:
    h, w = random_map.shape
    out = random_map.clone()
    for i in range(0, h, 1):
        for j in range(0, w, 96):
            block = random_map[i:i+1, j:j+96]
            mean_val = block.mean()
            out[i:i+1, j:j+96] = mean_val
    pooled_maps.append(out)

# 统一 color scale 的上下限
global_vmin = min(m.min().item() for m in pooled_maps)
global_vmax = max(m.max().item() for m in pooled_maps)

# 画图
plt.figure(figsize=(16, 4), dpi = 100)
for i, (out, title) in enumerate(zip(pooled_maps, titles)):
    ax = plt.subplot(1, 2, i + 1)
    sns.heatmap(out, fmt=".5f", cmap="YlGnBu", annot=False, ax=ax,
                vmin=global_vmin, vmax=global_vmax)  # 统一颜色范围

    h, w = out.shape
    xtick_locs = np.arange(0, w+1, 512)
    ytick_locs = np.arange(0, h+1, 2)
    ax.set_xticks(xtick_locs)
    ax.set_xticklabels(xtick_locs, rotation=45)
    ax.set_yticks(ytick_locs)
    ax.set_yticklabels(ytick_locs)
    ax.invert_yaxis()
    ax.set_title(title)

plt.tight_layout()
plt.show()

In [ ]:
normed_map = (attribution_map - attribution_map.mean(-1, keepdim=True)).div(torch.sqrt(attribution_map.var(-1, keepdim=True)+1e-8))
attribution_probs = normed_map.softmax(-1)
attribution_entropies = torch.einsum('blh,blh->bl', attribution_probs, -attribution_probs.log())
attribution_entropies = attribution_entropies.transpose(0,1).mean(-1)

fig, ax = plt.subplots(figsize=(8, 3))

# 绘制原始柱状图
ax.bar(range(len(attribution_entropies)),  attribution_entropies, width=1.0)


# 添加图形标题和标签
ax.set_xlabel('Layer')
ax.set_title('Normalized Entropies in Multiple Layers')

# 显示图形
plt.show()

In [ ]:
valid_neurons = attribution_map > 0.5 * attribution_map.amax(dim = (1,2), keepdim=True)
valid_neurons_num = valid_neurons.sum(dim = (0,2))

fig, ax = plt.subplots(figsize=(8, 3))

# 绘制原始柱状图
ax.bar(range(len(valid_neurons_num)),  valid_neurons_num, width=1.0)


# 添加图形标题和标签
ax.set_xlabel('Layer')
ax.set_title('Number of Knowledge Neurons in Multiple Layers')

# 显示图形
plt.show()

In [ ]:
import random
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from mpl_toolkits.axes_grid1 import make_axes_locatable

# 示例数据（请替换为你自己的 attribution_map）
# attribution_map = torch.randn(1500, 12, 1536)

# 预设索引
index1, index2 = 540, 1269
maps = [attribution_map[index1], attribution_map[index2]]
titles = [f"Figure (a): Attribution Map of Example {index1}", f"Figure (b): Attribution Map of Example {index2}"]

# 池化两个热图
pooled_maps = []
for random_map in maps:
    h, w = random_map.shape
    out = random_map.clone()
    for i in range(0, h, 1):
        for j in range(0, w, 96):
            block = random_map[i:i+1, j:j+96]
            mean_val = block.mean()
            out[i:i+1, j:j+96] = mean_val
    pooled_maps.append(out)

# 统一颜色范围
global_vmin = min(m.min().item() for m in pooled_maps)
global_vmax = max(m.max().item() for m in pooled_maps)


# 代码3：知识神经元数量


# 设置 2x2 网格
fig, axs = plt.subplots(1, 2, figsize=(16, 8), dpi=150)

# ===== 左上：热图（无 colorbar） =====
ax = axs[0, 0]
sns.heatmap(pooled_maps[0], fmt=".5f", cmap="YlGnBu", annot=False, ax=ax,
            cbar=False, vmin=global_vmin, vmax=global_vmax)
h, w = pooled_maps[0].shape
ax.set_xticks(np.arange(0, w + 1, 512))
ax.set_xticklabels(np.arange(0, w + 1, 512), rotation=45, fontsize = 14)
ax.set_yticks(np.arange(0, h + 1, 2))
ax.set_yticklabels(np.arange(0, h + 1, 2), fontsize = 14)
ax.invert_yaxis()
ax.set_title(titles[0], fontsize = 16)

# ===== 右上：热图（有 colorbar） =====
ax = axs[0, 1]
divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="4%", pad=0.1)  # colorbar 不挤压主图
sns.heatmap(pooled_maps[1], fmt=".5f", cmap="YlGnBu", annot=False, ax=ax,
            cbar=True, cbar_ax=cax, vmin=global_vmin, vmax=global_vmax)
h, w = pooled_maps[1].shape
ax.set_xticks(np.arange(0, w + 1, 512))
ax.set_xticklabels(np.arange(0, w + 1, 512), rotation=45, fontsize = 14)
ax.set_yticks(np.arange(0, h + 1, 2))
ax.set_yticklabels(np.arange(0, h + 1, 2), fontsize = 14)
ax.invert_yaxis()
ax.set_title(titles[1], fontsize = 16)


plt.tight_layout()
plt.show()


In [ ]:
normed_map = (attribution_map - attribution_map.mean(-1, keepdim=True)) / torch.sqrt(attribution_map.var(-1, keepdim=True) + 1e-8)
attribution_probs = normed_map.softmax(-1)
attribution_entropies = torch.einsum('blh,blh->bl', attribution_probs, -attribution_probs.log())
attribution_entropies = attribution_entropies.transpose(0, 1).mean(-1)


valid_neurons = attribution_map > 0.5 * attribution_map.amax(dim=(1, 2), keepdim=True)
valid_neurons_num = valid_neurons.sum(dim=(0, 2))

stds = attribution_map.std(-1).mean(0)

# 设置 2x2 网格
fig, axs = plt.subplots(1, 3, figsize=(15, 4), dpi=150)

ax = axs[0]
ax.bar(range(len(valid_neurons_num)), valid_neurons_num, width=1.0)
ax.set_xlabel('Layer', fontsize = 16)
ax.set_title('Figure (a): Number of Knowledge Neurons', fontsize = 16)
ax.tick_params(axis='both', labelsize=14)
# ===== 左下：熵柱状图 =====

ax = axs[1]
ax.bar(range(len(stds)), stds, width=1.0)
ax.set_xlabel('Layer', fontsize = 16)
ax.set_title('Figure (b): Standard Variance', fontsize = 16)
ax.tick_params(axis='both', labelsize=14)

ax = axs[2]
ax.bar(range(len(attribution_entropies)), attribution_entropies, width=1.0)
ax.set_xlabel('Layer', fontsize = 16)
ax.set_title('Figure (c): Normalized Entropy', fontsize = 16)
ax.tick_params(axis='both', labelsize=14)

# ===== 右下：知识神经元柱状图 =====




plt.tight_layout()
plt.show()